# Convolutional Neural Network

In [1]:
import torch, torchvision, re, os
import numpy as np
from PIL import Image
from collect_data import collect_data, train_model

DATA_DIR = '../data/images/climbs'

# Transform images to batch_size x channels x rows x columns
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

train_loader, val_loader = collect_data(DATA_DIR, preprocess)

In [2]:
# (for now) simple 2 layer Convolution with average pooling and 1 fully connected layer
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(3, 64, 3, padding='same')
        self.pool1 = torch.nn.AvgPool2d(2, 2)
        self.conv2 = torch.nn.Conv2d(64, 128, 3, padding='same')
        self.pool2 = torch.nn.AvgPool2d(2, 2)
        self.conv3 = torch.nn.Conv2d(128, 128, 3, padding='same')
        self.pool3 = torch.nn.AvgPool2d(2, 2)
        self.fc1 = torch.nn.Linear(100225, 4096) # 128 * 27 * 29 input features + 1 angle
        self.fc2 = torch.nn.Linear(4096, 23)
        self.flatten = torch.nn.Flatten()
        self.relu = torch.nn.ReLU()
    
    def forward(self, X, angles):
        c1 = self.conv1(X)
        r1 = self.relu(c1)
        p1 = self.pool1(r1)
        
        c2 = self.conv2(p1)
        r2 = self.relu(c2)
        p2 = self.pool2(r2)
        
        c3 = self.conv3(p2)
        r3 = self.relu(c3)
        p3 = self.pool3(r3)
        
        f = self.flatten(p3)
        f = torch.cat([f, angles], dim=1)
        z1 = self.fc1(f)
        a1 = self.relu(z1)
        
        y = self.fc2(a1)
        return y


In [3]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [4]:
model = CNN().to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 410843159


In [ ]:
import plotly.graph_objects as go
import time

# Hyperparameters
epsilon = 1E-6

# Use CCE loss
criterion = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=epsilon)

train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=3, checkpoint_path='kilter_cnn.pth')

Epoch [0/3], Train Loss: 3.13310, Val Loss: 3.12486 *, Train Acc: 0.02500, Validation Acc: 0.08750
